# Lab 15 — Ruido vs Fidelidad Guiada

Estudiamos cuantitativamente cómo el ruido degrada la fidelidad de circuitos cuánticos.
Comparamos diferentes modelos de ruido (despolarizante, phase-flip, amplitude-damping)
y analizamos cómo la profundidad del circuito amplifica el error.

**Objetivo**: calibrar la tolerancia al ruido para diferentes tipos de circuitos.

## 1. Modelos de error unitarios

Los tres canales básicos de error para un qubit son:
- **Despolarizante**: aplica X, Y, Z aleatoriamente con probabilidad p/3 cada uno
- **Phase-flip**: aplica Z con probabilidad p
- **Amplitude-damping**: decaimiento espontáneo |1⟩ → |0⟩ con tasa γ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import DensityMatrix, state_fidelity, Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import (
    NoiseModel, depolarizing_error, phase_damping_error, amplitude_damping_error
)

# Circuito de prueba: cadena de puertas H → fácil de analizar
def build_test_circuit(depth: int) -> QuantumCircuit:
    qc = QuantumCircuit(1)
    for _ in range(depth):
        qc.h(0)
    return qc

# Estado ideal para depth par: |0⟩; depth impar: |+⟩
def ideal_state(depth: int):
    qc = build_test_circuit(depth)
    return DensityMatrix(Statevector(qc))

print('Circuito de prueba (depth=3):')
print(build_test_circuit(3).draw('text'))

## 2. Fidelidad vs profundidad para canal despolarizante

Para un canal despolarizante de parámetro p, la fidelidad de una cadena de n puertas con el ideal
decae como F ≈ (1 - 4p/3)^n para p pequeño.

In [ ]:
def simulate_noisy(qc: QuantumCircuit, noise_model: NoiseModel) -> DensityMatrix:
    """Simula un circuito con ruido y retorna la DensityMatrix."""
    qc_dm = qc.copy()
    qc_dm.save_density_matrix()
    sim = AerSimulator(noise_model=noise_model)
    result = sim.run(qc_dm, shots=1).result()
    return DensityMatrix(result.data(0)['density_matrix'])

# Escaneo: fidelidad vs profundidad para diferentes tasas de error
depths = list(range(1, 21))
error_rates = [0.01, 0.05, 0.10]

fig, ax = plt.subplots(figsize=(9, 5))
for rate in error_rates:
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(rate, 1), ['h'])
    fids = []
    for d in depths:
        qc = build_test_circuit(d)
        dm_noisy = simulate_noisy(qc, nm)
        dm_ideal = ideal_state(d)
        fids.append(state_fidelity(dm_ideal, dm_noisy))
    ax.plot(depths, fids, 'o-', label=f'p={rate}')

ax.set_xlabel('Profundidad del circuito', fontsize=12)
ax.set_ylabel('Fidelidad', fontsize=12)
ax.set_title('Fidelidad vs Profundidad (canal despolarizante)', fontsize=13)
ax.legend(title='Tasa de error')
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Comparación de modelos de ruido

Los diferentes canales de error tienen distinta geometría en la esfera de Bloch:
- **Despolarizante**: contrae hacia el centro uniformemente
- **Phase-flip**: contrae el ecuador (desfase Z)
- **Amplitude-damping**: colapsa hacia |0⟩

In [ ]:
# Comparar modelos a profundidad fija
depth = 5
qc = build_test_circuit(depth)
dm_ideal = ideal_state(depth)
rate = 0.05

noise_models = {
    'Despolarizante': NoiseModel(),
    'Phase-flip': NoiseModel(),
    'Amplitude-damping': NoiseModel(),
}
noise_models['Despolarizante'].add_all_qubit_quantum_error(
    depolarizing_error(rate, 1), ['h'])
noise_models['Phase-flip'].add_all_qubit_quantum_error(
    phase_damping_error(rate), ['h'])
noise_models['Amplitude-damping'].add_all_qubit_quantum_error(
    amplitude_damping_error(rate), ['h'])

print(f'Comparación de modelos (depth={depth}, tasa={rate}):')
print(f'{"Modelo":<20} | Fidelidad | Pureza')
print('-' * 45)
print(f'{"Ideal":<20} | {1.0:.4f}    | {dm_ideal.purity():.4f}')
for name, nm in noise_models.items():
    dm_n = simulate_noisy(qc, nm)
    fid = state_fidelity(dm_ideal, dm_n)
    pur = dm_n.purity()
    print(f'{name:<20} | {fid:.4f}    | {pur:.4f}')

## 4. Umbral de fidelidad y profundidad de circuito

¿Cuántas puertas podemos aplicar antes de que la fidelidad caiga por debajo de 90%?
Esta es la **profundidad de circuito tolerada** para una tasa de error dada.

In [ ]:
threshold = 0.90
depths_fine = list(range(1, 100))

print('Profundidad máxima para fidelidad ≥ 90%:')
print(f'{"Tasa de error":<15} | Profundidad máxima')
print('-' * 35)
for rate in [0.001, 0.005, 0.01, 0.02, 0.05]:
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(rate, 1), ['h'])
    max_depth = 0
    for d in depths_fine:
        qc = build_test_circuit(d)
        dm_n = simulate_noisy(qc, nm)
        dm_i = ideal_state(d)
        if state_fidelity(dm_i, dm_n) >= threshold:
            max_depth = d
        else:
            break
    print(f'{rate:<15} | {max_depth}')

## 5. Resumen: impacto del ruido

| Canal | Efecto geométrico | Invariante dañado |
|-------|------------------|-----------------|
| Despolarizante | Contracción uniforme | Todo |
| Phase-flip | Contracción ecuatorial | Coherencia X,Y |
| Amplitude-damping | Deriva hacia |0⟩ | Población + coherencia |
| Bit-flip | Contracción de Z | Coherencia Z |

**Regla práctica**: para hardware cuántico actual (F_gate ≈ 99-99.5%), los circuitos útiles tienen profundidad < 100-200 puertas antes de que el ruido domine el resultado.

In [ ]:
# Proyección: profundidad máxima útil para hardware real
print('Proyección para hardware real (fidelidad puerta):')
print(f'{"F_gate":<10} | Tasa error (p) | Profundidad útil (F≥90%)')
print('-' * 55)
for f_gate in [0.999, 0.995, 0.99, 0.98, 0.95]:
    p_err = 1 - f_gate
    # Aproximación analítica: F(d) = (1 - 4p/3)^d ≥ 0.9
    d_max = int(np.log(0.9) / np.log(1 - 4*p_err/3))
    print(f'{f_gate:<10} | {p_err:.4f}          | ~{d_max}')